# 🕵️ ReAct: The Corporate Detective

Welcome to **Baskerville Tech**.

In this notebook, you will solve the same incident in two different ways:

1. **Act I — Dr. Watson (standard prompting style):** jump to a quick answer from narrative clues.
2. **Act II — Sherlock Holmes (ReAct):** reason step-by-step and call tools to gather evidence.

By the end, you should see why **Reason + Act** beats confident guessing for high-stakes business workflows.

## Notebook Roadmap

- **Part 1: Setup and Case File**
- **Part 2: Act I — Watson (fast but fallible)**
- **Part 3: ReAct Primer (Thought → Action → Observation)**
- **Part 3b: Interactive Detective Game (play by yourself)**
- **Part 4: Manual ReAct Investigation**
- **Part 5: Automated Sherlock Agent**
- **Part 6: Post-case Debrief and Reflection**

In [ ]:
# @title Setup: install the backend, fetch the case files, import dependencies { display-mode: "form" }
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any, Dict, List

try:
    import transformers
except ImportError:
    print("installing transformers (the language model this notebook runs on) ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])

REPO_OWNER = "ankitaghosh9"
REPO_NAME  = "ta"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if os.path.isdir(REPO_NAME):
        print("Repo already present — refreshing to latest…")
        res = subprocess.run(["git", "-C", REPO_NAME, "pull", "-q", url], capture_output=True, text=True)
        if res.returncode != 0:
            print("  (could not pull — using the existing copy)")
    else:
        print("Cloning the exercise repo…")
        res = subprocess.run(["git", "clone", "-q", url], capture_output=True, text=True)
        if res.returncode != 0:
            tail = (res.stderr.strip().splitlines() or ["(no message)"])[-1]
            print("  clone failed:", tail)

# Move to the workshop folder (the folder holding corpus/ and hf_llm.py) so imports resolve.
for _root in [
    os.path.join(REPO_NAME, "01_react_sherlock"),
    "01_react_sherlock",
    ".",
    os.path.dirname(os.getcwd()),
    os.getcwd(),
]:
    if os.path.exists(os.path.join(_root, "corpus", "case_data.json")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        f"01_react_sherlock case data not found — the clone of {REPO_OWNER}/{REPO_NAME} did not land "
        "(see the message above). Check the Colab runtime has network access, then re-run this cell.")

ROOT = Path.cwd()
ENV_DIR = str(ROOT)
sys.path.insert(0, ENV_DIR)

DATA_PATH = ROOT / "corpus" / "case_data.json"
with open(DATA_PATH, "r", encoding="utf-8") as f:
    CASE = json.load(f)

SUSPECTS = [s["name"] for s in CASE["suspects"]]

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:                             # torch missing, or a broken install
    DEVICE = "none"

print("Working directory:", os.getcwd(), "· helpers importable ✅")
print("case loaded from", ENV_DIR)
print("%d suspects, time window %s · case: %s"
      % (len(SUSPECTS), CASE["incident"]["time_window"], CASE["incident"]["name"]))
if DEVICE == "cuda":
    print("torch sees a GPU, so the language model later on will be comfortable.")
elif DEVICE == "cpu":
    print("torch sees no GPU. The model later on will run, slowly. Runtime, Change")
    print("runtime type, T4 takes a minute and is worth it.")
else:
    print("no torch here, so the model cannot load. There is no stand in: install torch.")

torch.manual_seed(42)

### Part 1b — Load a Hugging Face model

Watson and Sherlock both use a **local instruct model** from Hugging Face (`Qwen/Qwen2.5-0.5B-Instruct` by default).

- No API key required for the default model.
- First run downloads weights (~1 GB).
- GPU is faster; CPU works but is slower.

In [ ]:
import subprocess
import sys

def ensure_packages():
    required = ("torch", "transformers", "accelerate")
    missing = []
    for package in required:
        try:
            __import__(package)
        except ImportError:
            missing.append(package)
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "torch", "transformers", "accelerate"]
        )

ensure_packages()

In [ ]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from hf_llm import load_llm, watson_guess, run_react_agent, MODEL_ID

llm = load_llm(MODEL_ID)
print("Model ready:", MODEL_ID)

In [ ]:
# @title API Setup if needed {display-mode: "form"}
# # Setup OpenRouter API client
# # IMPORTANT: Add your OpenRouter API key as a Colab secret named 'OPENROUTER_API_KEY'
# # Go to the 🔑 icon in the left sidebar → Add a secret named OPENROUTER_API_KEY

# import os

# MODEL = "gemini-2.5-flash"
# CALLS_PER_MINUTE = 5      # what the free tier allows for this model. Raise it on a paid key.

# USE_REAL_LLM = False
# client = None

# _key = None
# try:
#     from google.colab import userdata
#     _key = userdata.get("GEMINI_API_KEY")
# except Exception:
#     _key = os.environ.get("GEMINI_API_KEY")       # so the same code runs outside Colab

# if _key:
#     from google import genai
#     client = genai.Client(api_key=_key)
#     USE_REAL_LLM = True
#     print("connected to %s, free tier budget %d calls a minute" % (MODEL, CALLS_PER_MINUTE))
# else:
#     print("no GEMINI_API_KEY found")
#     print("falling back to the offline stand in, everything below still works")

# try:
#     from google.genai import types
#     _CFG = types.GenerateContentConfig(temperature=0.0,
#                                        thinking_config=types.ThinkingConfig(thinking_budget=0))
# except Exception:
#     _CFG = None

# _call_times = []


# def _wait_for_quota():
#     """Never make more than CALLS_PER_MINUTE calls in any 60 seconds, so the free tier
#     never has to refuse us. Sleeps, out loud, when we are at the limit."""
#     while True:
#         now = time.time()
#         while _call_times and now - _call_times[0] > 60:
#             _call_times.pop(0)
#         if len(_call_times) < CALLS_PER_MINUTE:
#             _call_times.append(now)
#             return
#         wait = 60 - (now - _call_times[0]) + 1
#         print("   (free tier: %d calls a minute. Waiting %.0f s.)" % (CALLS_PER_MINUTE, wait))
#         time.sleep(wait)


# def _retry_seconds(message, fallback):
#     """The server usually tells us exactly how long to wait. Believe it."""
#     for pattern in (r"retryDelay['\"]?:\s*['\"]?(\d+(?:\.\d+)?)s", r"retry in (\d+(?:\.\d+)?)s"):
#         m = re.search(pattern, message)
#         if m:
#             return float(m.group(1)) + 1
#     return fallback


# def call_gemini(prompt, retries=5):
#     """One call, staying inside the free tier and backing off when told to."""
#     delay, cfg = 10.0, _CFG
#     for attempt in range(retries):
#         _wait_for_quota()
#         try:
#             r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
#             return (r.text or "").strip()
#         except Exception as e:
#             s = str(e)
#             if cfg is not None and ("thinking" in s.lower() or "INVALID_ARGUMENT" in s):
#                 cfg = None            # this model does not take the thinking setting
#                 continue
#             transient = any(t in s for t in ("429", "RESOURCE_EXHAUSTED", "503",
#                                              "UNAVAILABLE", "500", "INTERNAL"))
#             if not transient or attempt == retries - 1:
#                 raise
#             wait = _retry_seconds(s, delay)
#             print("   (the API asked us to slow down. Waiting %.0f s.)" % wait)
#             time.sleep(wait)
#             _call_times.clear()       # that wait cleared the window
#             delay = min(delay * 2, 60)
#     raise RuntimeError("gave up after %d attempts" % retries)

In [ ]:
# Quick narrative briefing
print("=== CASE BRIEF ===")
print(CASE["incident"]["summary"])
print("\n=== SUSPECT PROFILES ===")
for s in CASE["suspects"]:
    print(f"- {s['name']} ({s['role']}): {s['profile']}")

## Part 2 — Act I: Dr. Watson (Standard Prompting)

In this act, we intentionally simulate a **non-tool-using** assistant.

The point is not whether it can sound convincing; the point is whether it can **verify** claims.

A model without tool access often overweights dramatic narrative clues (motives, conflicts, personality) and underweights verifiable operational evidence.

In [ ]:
watson_result = watson_guess(CASE, llm)

print("Watson's accusation:", watson_result["culprit"])
print("Confidence:", watson_result["confidence"])
print("Reasoning:", watson_result["explanation"])
print("\n--- Raw model output ---")
print(watson_result.get("raw_response", ""))

### Why Watson Fails

Watson can only infer from the suspect bios. It does **not** check:

- Which credentials accessed sensitive systems in the breach window,
- Who was physically present,
- Who communicated suspicious intent,
- Who received suspicious payments.

That missing verification step is exactly what ReAct fixes.

## Part 3 — ReAct Primer

ReAct combines:

1. **Thought**: what do I need to learn next?
2. **Action**: which tool call can answer that?
3. **Observation**: what did the tool return?

The cycle repeats until enough evidence exists to make a justified decision.

In [ ]:
# Tool layer (mock corporate APIs)
def query_server_logs(time_window: str) -> Any:
    """Query server access logs for a 4-hour slot.

    Available slots: 00:00-04:00, 04:00-08:00, 08:00-12:00,
    12:00-16:00, 16:00-20:00, 20:00-00:00.

    The reported theft (00:30-02:00) falls in slot 00:00-04:00.
    """
    logs = CASE["server_logs"].get(time_window)
    if logs is None:
        return {
            "error": f"No logs for '{time_window}'.",
            "available_slots": list(CASE["server_logs"].keys()),
            "hint": "Reported theft is 00:30-02:00 → query '00:00-04:00'.",
        }
    return logs


def _hhmm_to_minutes(time_str: str) -> int:
    """Parse HH:MM into minutes since midnight."""
    parts = time_str.strip().split(":")
    if len(parts) != 2:
        raise ValueError(f"expected HH:MM, got {time_str!r}")
    hour, minute = int(parts[0]), int(parts[1])
    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError(f"out-of-range time: {time_str!r}")
    return hour * 60 + minute


def check_badge_swipes(time: str) -> Any:
    """Return who is still badge-IN (on premises) at HH:MM.

    Overnight shifts are supported when IN is later than OUT
    (e.g. Charlie: 22:58 IN, 07:12 OUT). Someone who OUT at exactly
    `time` is treated as already gone.

    Example: check_badge_swipes("01:00") during the theft window.
    """
    try:
        query = _hhmm_to_minutes(time)
    except ValueError:
        return {
            "error": f"Invalid time {time!r}. Use HH:MM, e.g. '01:00'.",
            "hint": "Try a time inside the breach window 00:30-02:00.",
        }

    active: List[str] = []
    for name, events in CASE["badge_swipes"].items():
        in_t = out_t = None
        for event in events:
            bits = event.split()
            if len(bits) != 2:
                continue
            stamp, kind = bits[0], bits[1].upper()
            if kind == "IN":
                in_t = _hhmm_to_minutes(stamp)
            elif kind == "OUT":
                out_t = _hhmm_to_minutes(stamp)
        if in_t is None or out_t is None:
            continue
        # Same-calendar-day shift vs overnight (IN after OUT on the clock).
        if in_t <= out_t:
            on_site = in_t <= query < out_t
        else:
            on_site = query >= in_t or query < out_t
        if on_site:
            active.append(name.title())
    return active


def inspect_work_emails(employee_name: str) -> List[str]:
    return CASE["emails"].get(employee_name, [])


def check_bank_records(employee_name: str) -> Dict[str, Any]:
    return CASE["bank_records"].get(employee_name, {})


TOOLS = {
    "query_server_logs": query_server_logs,
    "check_badge_swipes": check_badge_swipes,
    "inspect_work_emails": inspect_work_emails,
    "check_bank_records": check_bank_records,
}

print("Tools available:")
for name in TOOLS:
    print("-", name)
print("Log slots:", ", ".join(CASE["server_logs"].keys()))
print("Reported theft window:", CASE["incident"]["time_window"])

## Part 3b — 🎮 Interactive Detective Game

**Play the case yourself!** Use the embedded game below to:

1. Review all **six suspect profiles**
2. Query the **four corporate APIs** (enter inputs, see observations)
3. Build your investigation log
4. Submit your **final accusation**

> Tip: logs are stored in **4-hour slots** across the day. Start near the reported theft window (`query_server_logs("00:00-04:00")`, then cross-check badge swipes, emails, and bank records.


In [ ]:
import html as html_lib
from IPython.display import HTML, display

GAME_PATH = ROOT / "detective_game.html"
if not GAME_PATH.exists():
    raise FileNotFoundError(f"Game file not found: {GAME_PATH}")

EMBED_HEIGHT = 700
game_html = GAME_PATH.read_text(encoding="utf-8")
iframe = (
    '<iframe srcdoc="' + html_lib.escape(game_html, quote=True) + '" '
    'width="100%" height="' + str(EMBED_HEIGHT) + '" '
    'style="width:100%;height:' + str(EMBED_HEIGHT) + 'px;'
    'border:1px solid #2a3548;border-radius:10px;background:#0f1419;'
    'display:block;overflow:auto">'
    '</iframe>'
)
display(HTML(iframe))


In [ ]:
print("If the window below is empty, open folder icon on the sidebar and download the file:")
print(" ", GAME_PATH.resolve())
print("Then open detective_game.html for the full-page version.")

## Part 5 — Build ReAct from Scratch

In the game window, you drove **Thought → Action → Observation** by hand.
Now the **LLM** chooses each Action; you wire the loop yourself.

You need three pieces:

1. **Ask the model** for the next Thought + Action + Action Input (or Final Answer)
2. **Execute** that Action with our tools and capture the Observation
3. **Loop**: append Observation to a scratchpad, ask again, stop when the model gives a Final Answer


### 5.1 — Ask the LLM what to do next

The model must reply in ReAct format — either a tool call, or a final accusation:

```text
Thought: ...
Action: <tool_name>
Action Input: <plain string>
```

or

```text
Thought: ...
Final Answer: <suspect name>
```


In [ ]:
# 5.1 — Prompt the LLM and parse its next step
from hf_llm import format_suspects, normalize_culprit, parse_react_step

MAX_REACT_STEPS = 10

REQUIRED_BEFORE_VERDICT = {"query_server_logs", "check_badge_swipes", "inspect_work_emails", "check_bank_records"}

# Keep tool docs + ReAct format + investigation policy HERE (not only in hf_llm.py)
# so the notebook controls how Sherlock is guided.
TOOL_DESCRIPTIONS = """
- query_server_logs(time_window): server access in a 4-hour slot ONLY
  (valid: 00:00-04:00, 04:00-08:00, 08:00-12:00, 12:00-16:00, 16:00-20:00, 20:00-00:00)
- check_badge_swipes(time): who is still badge-IN at HH:MM in 24-hour format (e.g. 01:00)
- inspect_work_emails(employee_name): recent email snippets for one employee
- check_bank_records(employee_name): wage vs deposit/withdraw + flagged wire
"""

REACT_FORMAT = """
Respond with exactly ONE of these formats per turn — never both.
Never invent Observations.

Tool call:
Thought: <your reasoning>
Action: <exact lowercase tool name>
Action Input: <plain string, e.g. 01:00 or 04:00-08:00 or Alice>

Verdict:
Thought: <your reasoning>
Final Answer: <one suspect name only, e.g. Alice>

Rules:
- Thought, then either Action + Action Input OR Final Answer — never both.
- After Action Input, stop.
- Action Input must be a plain string, not JSON.
"""


def build_system_prompt() -> str:
    return (
        "You are Sherlock Holmes, a ReAct investigator at Baskerville Tech.\n"
        "Gather evidence with tools before accusing anyone.\n"
        "Each reply is ONE step only.\n"
        f"Available tools:\n{TOOL_DESCRIPTIONS}\n"
        f"{REACT_FORMAT}"
    )


def build_user_prompt(scratchpad: str) -> str:
    log = scratchpad.strip() or (
        '(none yet — start with check_badge_swipes("01:00"),'
         'then query server logs, query emails, and check bank records)'
    )
    return (
        f"Incident: {CASE['incident']['summary']}\n"
        f"Breach window: {CASE['incident']['time_window']}\n\n"
        f"Suspects:\n{format_suspects(CASE)}\n\n"
        f"Investigation log so far:\n{log}\n\n"
        "Reply with your next single step only "
        "(Thought + Action + Action Input, or Thought + Final Answer)."
    )


def llm_decide(scratchpad: str) -> dict:
    """Ask the LLM for the next ReAct step and parse Action / Action Input / Final Answer."""
    raw = llm.generate(build_system_prompt(), build_user_prompt(scratchpad))
    step = parse_react_step(raw)
    step["raw"] = raw
    return step


# Sanity check: one undecided step (no tools run yet)
_demo = llm_decide("")
print("Demo Thought:", _demo.get("thought"))
print("Demo Action:", _demo.get("action"))
print("Demo Action Input:", _demo.get("action_input"))
print("Demo Final Answer:", _demo.get("final_answer"))
print("--- raw ---")
print(_demo["raw"])

### 5.2 — Execute the chosen action

Whatever the model asked for, look it up in `TOOLS`, run it, and return the Observation.
Unknown tools become an error Observation (fed back to the model on the next turn).


In [ ]:
# 5.2 — Run one tool call and return an Observation
def execute_action(action: str | None, action_input: Any) -> Any:
    """Execute the LLM's chosen tool call. Always returns something the model can read."""
    if not action or action not in TOOLS:
        return {
            "error": f"Invalid or missing action: {action!r}.",
            "available_tools": list(TOOLS),
        }
    try:
        return TOOLS[action](action_input)
    except Exception as exc:  # noqa: BLE001 — surface tool errors as observations
        return {"error": str(exc)}


# Sanity check against a known slot
print(execute_action("query_server_logs", "00:00-04:00"))
print(execute_action("not_a_real_tool", "Alice"))


### 5.3 — The ReAct loop

Until the model issues a **Final Answer** (or you hit the step budget):

1. `llm_decide(scratchpad)` → Thought / Action / Action Input
2. `execute_action(...)` → Observation
3. Append Thought, Action, Action Input, Observation to the scratchpad
4. Repeat — the Observation is now part of the next prompt


In [ ]:
# 5.3 — Full ReAct loop: decide → act → observe → repeat
def run_react_from_scratch(max_steps: int = MAX_REACT_STEPS) -> dict:
    suspects = [s["name"] for s in CASE["suspects"]]
    scratchpad = ""
    trace = []
    culprit = None
    tools_used: set[str] = set()

    for step_i in range(1, max_steps + 1):
        decision = llm_decide(scratchpad)
        thought = decision.get("thought") or "Continuing investigation."
        action = decision.get("action")
        action_input = decision.get("action_input")
        if action_input is not None:
            action_input = action_input.lower()
        final_answer = decision.get("final_answer")

        print(f"\n=== Step {step_i} ===")
        print("Thought:", thought)

        # Soft gate: notebook policy requires bank motive before a verdict
        if final_answer:
            missing = REQUIRED_BEFORE_VERDICT - tools_used
            if missing:
                observation = {
                    "error": "Premature Final Answer rejected by notebook policy.",
                    "missing_tools": sorted(missing),
                    "hint": (
                        "Policy: call check_bank_records and inspect_work_emails on your leading suspect(s) "
                        "Then Final Answer with a Thought that CONFIRMS opportunity "
                        "+ suspicious server action + financial motive."
                    ),
                }
                print("Action: <blocked Final Answer>")
                print("Observation:", observation)
                scratchpad += (
                    f"\nThought: {thought}\n"
                    f"Action: Final Answer\n"
                    f"Action Input: {final_answer}\n"
                    f"Observation: {observation}\n"
                )
                decision["observation"] = observation
                trace.append(decision)
                continue

            culprit = normalize_culprit(final_answer, suspects) or final_answer
            print("Final Answer:", culprit)
            decision["observation"] = {"final_answer": final_answer}
            trace.append(decision)
            scratchpad += f"\nThought: {thought}\nFinal Answer: {final_answer}\n"
            break

        print("Action:", action)
        print("Action Input:", action_input)

        observation = execute_action(action, action_input)
        print("Observation:", observation)

        if action and not (isinstance(observation, dict) and "error" in observation):
            tools_used.add(action)

        scratchpad += (
            f"\nThought: {thought}\n"
            f"Action: {action}\n"
            f"Action Input: {action_input}\n"
            f"Observation: {observation}\n"
        )
        decision["observation"] = observation
        trace.append(decision)

    if culprit is None:
        print("\n=== Forced final accusation ===")
        forced_prompt = (
            build_user_prompt(scratchpad)
            + "\n\nChecklist time is over. Accuse exactly one suspect.\n"
            "Thought must confirm opportunity + suspicious action + motive "
            "from the investigation log (do not exonerate then accuse).\n"
            "Reply with Thought + Final Answer only."
        )
        raw = llm.generate(build_system_prompt(), forced_prompt, max_new_tokens=160)
        decision = parse_react_step(raw)
        decision["raw"] = raw
        final_answer = decision.get("final_answer") or raw
        culprit = normalize_culprit(final_answer, suspects) or suspects[0]
        print("Thought:", decision.get("thought"))
        print("Final Answer:", culprit)
        decision["observation"] = {"final_answer": final_answer}
        trace.append(decision)

    return {"culprit": culprit, "trace": trace, "scratchpad": scratchpad}


sherlock_result = run_react_from_scratch()
print("\nSherlock's culprit:", sherlock_result["culprit"])
print("Ground Truth:", CASE["ground_truth"]["culprit"])
print("Match:", sherlock_result["culprit"] == CASE["ground_truth"]["culprit"])


In [ ]:
# Optional: inspect the full raw model outputs for each step
for i, step in enumerate(sherlock_result["trace"], start=1):
    print(f"\n--- Step {i} raw ---")
    print(step.get("raw", ""))


## Part 6 — Debrief

### What changed between Watson and Sherlock?

- **Watson** optimized for a plausible story from profiles alone.
- **Sherlock** optimized for verifiable evidence via tool calls.

### Core concept takeaway

ReAct is not just "better prompting." It changes the operating model:

- from one-shot guessing
- to iterative investigation
- with tool-grounded observations at every step.

### Extension challenges

1. Add noisy or contradictory signals to test robustness.
2. Add a cost budget and force Sherlock to minimize tool calls.
3. Swap `Qwen/Qwen2.5-0.5B-Instruct` for a larger model and compare trace quality.